# KBR Pajak Fase 4 — Ekstraksi Aturan Hukum (`T2_RULE`) via vLLM (Colab Pro A100)

**Persiapan sebelum Run:**
1. Di Google Drive buat folder **`kbr_pajak_fase4/`** lalu upload ke dalamnya:
   - `all_t2_chunks.jsonl` (dari `fatih/data/chunks/pending/all_t2_chunks.jsonl`)
   - `gold_standard_template.jsonl` (dari `fatih/data/gold_standard_template.jsonl`)
2. Pilih **Runtime → Change runtime type → A100 GPU**.
3. Jalankan Cell 1–5 (Sanity Check 36 chunk dulu ~2 menit), cek hasilnya, baru jalankan Cell 6 untuk ekstraksi penuh 5.401 chunk (resumable, otomatis lanjut kalau disconnect).


In [ ]:
# 1. Mount Google Drive & cek spesifikasi GPU
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/kbr_pajak_fase4')
RES_DIR = DRIVE_DIR / 'llm_results'
RES_DIR.mkdir(parents=True, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# 2. Install vLLM & dependensi CUTLASS/CUDA (jalankan sekali per sesi, lalu Restart session bila diminta)
!pip install -q vllm nvidia-cutlass-dsl cuda-python


In [ ]:
# 3. Konfigurasi model & load ke vLLM (Colab-safe)
import os
import subprocess

# WAJIB diset paling awal sebelum import torch / vllm
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

try:
    _vram_mib = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()[0]
    vram_gb = int(_vram_mib) / 1024.0
except Exception:
    vram_gb = 0.0
print(f"GPU VRAM terdeteksi: {vram_gb:.1f} GB")

USE_72B_IF_80GB = False
if USE_72B_IF_80GB and vram_gb >= 75:
    MODEL_ID = "Qwen/Qwen2.5-72B-Instruct-AWQ"
else:
    MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"

PROMPT_VERSION = "T2_v1"
print("Model terpilih:", MODEL_ID)

from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    max_model_len=8192,
    gpu_memory_utilization=0.80,
    max_num_seqs=16,
    enforce_eager=True,
)
tokenizer = llm.get_tokenizer()
sampling_params = SamplingParams(temperature=0.0, max_tokens=4096)


In [ ]:
# 4. Definisi Prompt T2_v1 & fungsi ekstraksi batch
import json, re, time

SYSTEM_PROMPT = '''Anda adalah ekstraktor aturan hukum perpajakan Indonesia yang sangat teliti.
Tugas: ekstrak aturan hukum atomik dari teks pasal/ayat peraturan yang diberikan ke dalam JSON valid.

ATURAN WAJIB:
1. Keluarkan HANYA objek JSON valid tanpa teks pembuka/penutup atau markdown fence.
2. Jangan mengarang angka, tarif, tanggal, syarat, atau dasar hukum yang TIDAK tertulis persis di teks input.
3. Setiap aturan wajib menyertakan "source_quote" berupa kutipan SUBSTRING PERSIS dari teks input.
4. Pisahkan kondisi (condition), kewajiban (obligation), hak (right), larangan (prohibition), pengecualian (exception), konsekuensi/sanksi (consequence), tenggat (deadline), dan tarif/nilai (rate_or_value).
5. Nilai "rule_type" harus salah satu dari:
   "eligibility", "obligation", "right", "prohibition", "exception", "sanction", "deadline", "rate", "definition", "procedure", "other".
6. Jika pasal tidak memuat aturan substantif/definisi, keluarkan {"found": false, "rules": []}.
7. JANGAN mengubah atau mengonversi angka. Tulis persis seperti di teks input, termasuk "Rp4,8 miliar" (bukan 4800000000), "0,5%" (bukan 0.5), dan "3Oo/o" jika memang demikian.
8. Jika teks berupa tabel, JANGAN menyusun ulang baris/kolom; kutip blok tabel apa adanya.
9. "source_quote" harus substring persis; jangan memperbaiki ejaan, tanda baca, atau format angka.
10. Jika pasal memiliki daftar huruf a/b/c, kutip satu kalimat utuh atau hanya poin huruf yang relevan; jangan menggabungkan pembuka dengan huruf b/c sambil melompati huruf a.

SKEMA JSON:
{
  "found": true,
  "rules": [
    {
      "rule_type": "rate",
      "subject": "subjek hukum jika tertulis, atau null",
      "condition": [{"attribute": "...", "operator": "==|!=|<=|>=|<|>|in", "value": "...", "raw": "..."}],
      "obligation": ["..."],
      "right": ["..."],
      "prohibition": ["..."],
      "exception": ["..."],
      "consequence": null,
      "deadline": null,
      "rate_or_value": [{"value": 11, "unit": "%", "raw": "11% (persis seperti di teks)"}],
      "cross_references": ["rujukan pasal/UU persis sebagaimana tertulis"],
      "source_quote": "kutipan persis dari input",
      "effective_period_hint": null,
      "confidence": 0.95
    }
  ]
}'''

def build_prompt(chunk):
    ctx = chunk.get("context") or {}
    user_msg = (
        f"Dokumen: {chunk['document_id']} | Locator: {chunk['source_locator']} | Domain: {chunk.get('domain')}\n"
        f"Judul: {ctx.get('judul', '')}\n"
        f"Bagian: {ctx.get('pasal_induk', '')} ({ctx.get('sub_bagian') or ''})\n\n"
        f"Teks:\n{chunk['text']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_json_output(raw):
    s = (raw or "").strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    for cand in (s, re.sub(r"[\x00-\x1f]+", " ", s)):
        try:
            return json.loads(cand, strict=False), None
        except Exception:
            pass
    m = re.search(r"\{.*\}", s, re.S)
    if m:
        blk = m.group(0)
        for cand in (blk, re.sub(r"[\x00-\x1f]+", " ", blk)):
            try:
                return json.loads(cand, strict=False), None
            except Exception as e2:
                last_err = str(e2)
        return None, last_err
    return None, "no_json_object"

def run_batch(chunks_list, out_jsonl, batch_size=32, skip_tables=False):
    # Chunk tabel jangan diekstrak otomatis; simpan untuk review manusia.
    if skip_tables:
        table_chunks = [c for c in chunks_list if (c.get("context") or {}).get("is_table")]
        if table_chunks:
            table_out = out_jsonl.parent / "tables_for_review.jsonl"
            existing = set()
            if table_out.exists():
                for line in table_out.read_text(encoding="utf-8").splitlines():
                    if line.strip():
                        try:
                            existing.add(json.loads(line)["chunk_id"])
                        except Exception:
                            pass
            with table_out.open("a", encoding="utf-8") as tf:
                for c in table_chunks:
                    if c["chunk_id"] not in existing:
                        tf.write(json.dumps(c, ensure_ascii=False) + "\n")
            print(f"Chunk tabel dipisahkan ke review: {len(table_chunks)}")
        chunks_list = [c for c in chunks_list if not (c.get("context") or {}).get("is_table")]

    done_ids = set()
    if out_jsonl.exists():
        for line in out_jsonl.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    done_ids.add(json.loads(line)["chunk_id"])
                except Exception:
                    pass
    todo = [c for c in chunks_list if c["chunk_id"] not in done_ids]
    print(f"Sudah selesai: {len(done_ids)} | Sisa antrean: {len(todo)}")

    ok_cnt, err_cnt = 0, 0
    t0 = time.time()
    with out_jsonl.open("a", encoding="utf-8") as fout:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            prompts = [build_prompt(c) for c in batch]
            tb = time.time()
            outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
            dt = time.time() - tb
            for c, out in zip(batch, outputs):
                raw_text = out.outputs[0].text
                parsed, perr = parse_json_output(raw_text)
                if perr is None:
                    ok_cnt += 1
                else:
                    err_cnt += 1
                rec = {
                    "result_key": f"{c['chunk_id']}|T2_RULE|{PROMPT_VERSION}|{MODEL_ID}",
                    "chunk_id": c["chunk_id"],
                    "task_type": "T2_RULE",
                    "model_id": MODEL_ID,
                    "prompt_version": PROMPT_VERSION,
                    "raw_output": raw_text,
                    "parsed_output": parsed,
                    "parse_error": perr,
                    "latency_s": round(dt / max(1, len(batch)), 3),
                    "tokens_in": len(out.prompt_token_ids),
                    "tokens_out": len(out.outputs[0].token_ids),
                }
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fout.flush()
            done_now = i + len(batch)
            rate = done_now / max(0.1, time.time() - t0)
            print(f"[{done_now}/{len(todo)}] ok={ok_cnt} err={err_cnt} ({rate:.1f} chunk/dtk)")


In [ ]:
# 5. SANITY CHECK (36 chunk gold_standard_template.jsonl -> ~1-2 menit)
gold_path = DRIVE_DIR / "gold_standard_template.jsonl"
gold_chunks = [json.loads(l) for l in gold_path.read_text(encoding="utf-8").splitlines() if l.strip()]
# Lengkapi context minimal jika dari template
for gc in gold_chunks:
    gc.setdefault("context", {"judul": gc["document_id"], "pasal_induk": gc["source_locator"]})

sanity_out = RES_DIR / "sanity_36_v5.jsonl"
run_batch(gold_chunks, sanity_out, batch_size=18)

# Tampilkan 2 sampel hasil sanity check
for line in sanity_out.read_text(encoding="utf-8").splitlines()[:2]:
    r = json.loads(line)
    print(r["chunk_id"], "parse_error=", r["parse_error"], "rules=", len((r.get("parsed_output") or {}).get("rules", [])))


In [ ]:
# 6. EKSTRAKSI MASSAL PENUH (5.509 chunk)
# Jalankan cell ini setelah hasil sanity_36.jsonl dicek dan memuaskan.
all_path = DRIVE_DIR / "all_t2_chunks.jsonl"
all_chunks = [json.loads(l) for l in all_path.read_text(encoding="utf-8").splitlines() if l.strip()]
full_out = RES_DIR / "t2_results_v2.jsonl"
run_batch(all_chunks, full_out, batch_size=32, skip_tables=False)
